In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
import torch, math
from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer, Trainer,
    TrainingArguments, DataCollatorForLanguageModeling, set_seed
)
from datasets import Dataset

set_seed(42)

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

In [ ]:
def generate_text(model, tokenizer, prompt, max_length=100):
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
review_prompts = [
    'This product is',
    'I bought this phone and',
    'The quality of this item',
]

print('=== BASELINE REVIEWS (Before Fine-Tuning) ===')

baseline = {}
for p in review_prompts:
    baseline[p] = generate_text(model, tokenizer, p)
    print(f'Prompt: {p}\nOutput: {baseline[p]}\n')

In [ ]:
corpus = [
    'this phone has an amazing battery life and the camera quality is outstanding for the price.',
    'i bought this laptop for college and it handles all my assignments and coding projects perfectly.',
    'the sound quality of these headphones is incredible with deep bass and clear vocals.',
    'this smartwatch tracks my steps accurately and the heart rate monitor is very reliable.',
    'great wireless earbuds with noise cancellation that blocks out all background sound.',
    'the keyboard feels very comfortable for long typing sessions and the backlight is a nice touch.',
    'this portable charger saved me during travel and it charges my phone three times on a single charge.',
    'the tablet screen is bright and colorful which makes watching movies a great experience.',
    'i love this fitness tracker because it motivates me to reach my daily exercise goals.',
    'this bluetooth speaker is compact but delivers surprisingly loud and clear audio.',
    'the delivery was fast and the product was packed securely with no damage at all.',
    'excellent value for money and the build quality feels premium despite the affordable price.',
    'the customer service team was very helpful when i had questions about the product features.',
    'this camera takes stunning photos in low light and the video recording quality is very smooth.',
    'i have been using this product for three months and it still works perfectly like day one.',
    'the design is sleek and modern and it looks great on my desk next to my other gadgets.',
    'easy to set up right out of the box and the instructions were clear and simple to follow.',
    'highly recommend this product to anyone looking for quality and reliability at a fair price.',
    'the software updates keep adding new features which makes this purchase even more worthwhile.',
    'best purchase i made this year and i would definitely buy from this brand again.',
]

dataset = Dataset.from_dict({'text': corpus})

tokenized = dataset.map(
    lambda x: tokenizer(x['text'], truncation=True, max_length=128, padding='max_length'),
    batched=True,
    remove_columns=['text']
)

split = tokenized.train_test_split(test_size=0.15, seed=42)

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir='./gpt2-reviews',
    num_train_epochs=15,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy='epoch',
    logging_steps=10,
    save_strategy='no',
    fp16=torch.cuda.is_available(),
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    data_collator=data_collator,
)

trainer.train()

In [ ]:
eval_res = trainer.evaluate()
print(f'Perplexity: {math.exp(eval_res["eval_loss"]):.2f}')

print('\n=== FINE-TUNED REVIEWS (After Fine-Tuning) ===')

for p in review_prompts:
    ft_out = generate_text(model, tokenizer, p)
    print(f'Prompt: {p}')
    print(f'  Baseline:   {baseline[p][:120]}')
    print(f'  Fine-Tuned: {ft_out[:120]}\n')